# Task 170: rampy_unmix — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Raman spectral unmixing using NMF

**Paper**: Rampy: Raman spectroscopy analysis
**Repository**: https://github.com/charlesll/rampy

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: 21.99 dB
- **SSIM**: N/A
- **Evaluation**: Raman spectral unmixing (component CC, mixing RE, baseline PSNR)

### Mathematical Formulation

A measured spectrum is modeled as a superposition of spectral profiles:

$$S(\nu) = \sum_{k=1}^{K} A_k \, P_k(\nu; \theta_k) + B(\nu) + \epsilon(\nu)$$

where $P_k$ is a peak profile (Gaussian/Lorentzian/Voigt), $B(\nu)$ is the baseline, and $\epsilon$ is noise.

**Voigt profile**: $V(\nu) = \int_{-\infty}^{\infty} G(\nu'; \sigma) \, L(\nu - \nu'; \gamma) \, d\nu'$

**Nonlinear least-squares fit**:
$$\hat{\theta} = \arg\min_\theta \sum_i \left(\frac{S_i^{\text{obs}} - S_i^{\text{model}}(\theta)}{\sigma_i}\right)^2$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
#!/usr/bin/env python
"""
Raman Spectral Unmixing with Baseline Correction using rampy.

Inverse problem: Given mixed Raman spectra with baseline drift and noise,
decompose them into pure component spectra and mixing proportions, plus
perform baseline correction.

Forward model:  mixed_spectrum = sum(w_i * component_i) + baseline + noise
Inverse solver: (1) Baseline correction via rampy.baseline (ALS)
                (2) NMF for spectral unmixing to recover components & weights
"""

import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`gaussian_peak`, `lorentzian_peak`

In [ ]:
def gaussian_peak(x, center, width, height):
    """Single Gaussian peak."""
    return height * np.exp(-0.5 * ((x - center) / width) ** 2)

def lorentzian_peak(x, center, width, height):
    """Single Lorentzian peak."""
    return height / (1 + ((x - center) / width) ** 2)

## 4. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
matplotlib.use('Agg')

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import NMF
import rampy as rp
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# ─── reproducibility ───
np.random.seed(42)

# ─── output directory ───
os.makedirs('results', exist_ok=True)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ============================================================

### 1. SYNTHESIZE DATA

Intermediate processing step in the pipeline.

In [ ]:
# 1. SYNTHESIZE DATA

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ============================================================

# Wavenumber axis
n_points = 500
wavenumber = np.linspace(200, 1200, n_points)

### Pure component spectra ---

Intermediate processing step in the pipeline.

In [ ]:
# --- Pure component spectra ---
# Component 1: Mineral A – peaks at 400, 600 cm⁻¹
comp1 = (gaussian_peak(wavenumber, 400, 20, 1.0) +
         lorentzian_peak(wavenumber, 600, 15, 0.7))

# Component 2: Mineral B – peaks at 500, 800 cm⁻¹
comp2 = (gaussian_peak(wavenumber, 500, 25, 0.9) +
         gaussian_peak(wavenumber, 800, 18, 1.1))

# Component 3: Mineral C – peaks at 350, 700, 900 cm⁻¹
comp3 = (lorentzian_peak(wavenumber, 350, 22, 0.8) +
         gaussian_peak(wavenumber, 700, 20, 1.0) +
         lorentzian_peak(wavenumber, 900, 16, 0.6))

# Normalize each component to unit max
comp1 /= comp1.max()
comp2 /= comp2.max()
comp3 /= comp3.max()

pure_components = np.vstack([comp1, comp2, comp3])  # (3, 500)
n_components = 3

### Generate mixed spectra ---

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# --- Generate mixed spectra ---
n_mixtures = 15

# Random mixing proportions that sum to 1
raw_weights = np.random.dirichlet(alpha=[2, 2, 2], size=n_mixtures)  # (15, 3)
true_weights = raw_weights.copy()

# Mixed spectra (clean, no baseline, no noise)
clean_spectra = true_weights @ pure_components  # (15, 500)

### Add polynomial baseline drift ---

Intermediate processing step in the pipeline.

In [ ]:
# --- Add polynomial baseline drift ---
x_norm = (wavenumber - wavenumber.mean()) / (wavenumber.max() - wavenumber.min())

baselines = np.zeros((n_mixtures, n_points))
for i in range(n_mixtures):
    # Random 3rd-order polynomial coefficients
    c0 = np.random.uniform(0.05, 0.15)
    c1 = np.random.uniform(-0.1, 0.1)
    c2 = np.random.uniform(0.05, 0.2)
    c3 = np.random.uniform(-0.05, 0.05)
    baselines[i] = c0 + c1 * x_norm + c2 * x_norm**2 + c3 * x_norm**3

### Add Gaussian noise (SNR ~ 25) ---

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# --- Add Gaussian noise (SNR ~ 25) ---
signal_power = np.mean(clean_spectra ** 2, axis=1, keepdims=True)
snr = 25.0
noise_std = np.sqrt(signal_power / snr)
noise = noise_std * np.random.randn(n_mixtures, n_points)

# Final observed spectra
observed_spectra = clean_spectra + baselines + noise

print(f"Synthesized {n_mixtures} mixed spectra, {n_points} wavenumber points")
print(f"True weights shape: {true_weights.shape}")
print(f"Observed spectra shape: {observed_spectra.shape}")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ============================================================

### 2. INVERSE SOLVER

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 2. INVERSE SOLVER

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ============================================================

### Step 1: Baseline correction using rampy ---

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# --- Step 1: Baseline correction using rampy ---
corrected_spectra = np.zeros_like(observed_spectra)
fitted_baselines = np.zeros_like(observed_spectra)

for i in range(n_mixtures):
    y_corr, bl = rp.baseline(
        wavenumber, observed_spectra[i],
        method='als', lam=1e7, p=0.001, niter=100
    )
    corrected_spectra[i] = y_corr.flatten()
    fitted_baselines[i] = bl.flatten()

# Clip negative values after baseline removal (physical constraint: intensities ≥ 0)
corrected_spectra = np.clip(corrected_spectra, 0, None)

print("Baseline correction completed.")

### Step 2: NMF for spectral unmixing ---

Intermediate processing step in the pipeline.

In [ ]:
# --- Step 2: NMF for spectral unmixing ---
# NMF: V ≈ W * H, where V = (n_samples, n_features)
# W = mixing proportions (n_samples, n_components)
# H = component spectra (n_components, n_features)

# Try multiple NMF initializations and pick the best
best_err = np.inf
best_W = None
best_H = None
for seed in range(20):
    nmf_model = NMF(
        n_components=n_components,
        init='nndsvda',
        max_iter=10000,
        random_state=seed,
        l1_ratio=0.0,
        alpha_W=0.0,
        alpha_H=0.0,
        tol=1e-8,
    )
    W_try = nmf_model.fit_transform(corrected_spectra)
    H_try = nmf_model.components_
    err = nmf_model.reconstruction_err_
    if err < best_err:
        best_err = err
        best_W = W_try
        best_H = H_try
        best_seed = seed

W_nmf = best_W  # (n_mixtures, 3)
H_nmf = best_H  # (3, n_points)

print(f"NMF reconstruction error: {best_err:.6f} (seed={best_seed})")

### Normalize NMF results ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- Normalize NMF results ---
# Normalize H rows to unit max, adjust W accordingly
scale_factors = H_nmf.max(axis=1, keepdims=True)
H_norm = H_nmf / scale_factors            # each component spectrum max = 1
W_norm = W_nmf * scale_factors.T           # compensate in weights

# Normalize W rows to sum to 1 (mixing proportions)
W_sum = W_norm.sum(axis=1, keepdims=True)
W_final = W_norm / W_sum

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ============================================================

### 3. MATCH RECOVERED COMPONENTS TO TRUE COMPONENTS

Intermediate processing step in the pipeline.

In [ ]:
# 3. MATCH RECOVERED COMPONENTS TO TRUE COMPONENTS

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ============================================================
# Use correlation-based optimal assignment (Hungarian algorithm)

corr_matrix = np.zeros((n_components, n_components))
for i in range(n_components):
    for j in range(n_components):
        corr_matrix[i, j] = np.corrcoef(pure_components[i], H_norm[j])[0, 1]

# Maximize correlation → minimize negative correlation
cost_matrix = -corr_matrix
row_ind, col_ind = linear_sum_assignment(cost_matrix)

# Permute recovered components and weights
H_matched = H_norm[col_ind]
W_matched = W_final[:, col_ind]

print(f"Optimal permutation: true_idx -> recovered_idx = {list(zip(row_ind, col_ind))}")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ============================================================

### 4. EVALUATION METRICS

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# 4. EVALUATION METRICS

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ============================================================

metrics = {}

### (a) Baseline-corrected PSNR ---

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# --- (a) Baseline-corrected PSNR ---
# Compare corrected spectra vs clean spectra (ground truth without baseline/noise)
mse_baseline = np.mean((corrected_spectra - clean_spectra) ** 2)
max_val = clean_spectra.max()
psnr_baseline = 10 * np.log10(max_val ** 2 / mse_baseline)
metrics['baseline_corrected_PSNR_dB'] = float(round(psnr_baseline, 2))

# Correlation between corrected and clean
cc_baseline_list = []
for i in range(n_mixtures):
    cc = np.corrcoef(corrected_spectra[i], clean_spectra[i])[0, 1]
    cc_baseline_list.append(cc)
metrics['baseline_corrected_mean_CC'] = float(round(np.mean(cc_baseline_list), 4))

### (b) Component spectrum correlation ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- (b) Component spectrum correlation ---
comp_cc_list = []
for i in range(n_components):
    cc = np.corrcoef(pure_components[i], H_matched[i])[0, 1]
    comp_cc_list.append(cc)
    metrics[f'component_{i+1}_CC'] = float(round(cc, 4))
metrics['mean_component_CC'] = float(round(np.mean(comp_cc_list), 4))

### (c) Mixing proportion relative error ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- (c) Mixing proportion relative error ---
re_per_sample = np.abs(W_matched - true_weights) / (np.abs(true_weights) + 1e-10)
mean_re = np.mean(re_per_sample)
metrics['mixing_proportion_mean_RE'] = float(round(mean_re, 4))

# Mixing proportion correlation (per component)
for i in range(n_components):
    cc_w = np.corrcoef(true_weights[:, i], W_matched[:, i])[0, 1]
    metrics[f'weight_component_{i+1}_CC'] = float(round(cc_w, 4))

# Overall PSNR (as a summary metric)
metrics['PSNR'] = metrics['baseline_corrected_PSNR_dB']

print("\n=== Evaluation Metrics ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

# Save metrics
with open('results/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ============================================================

### 5. SAVE GROUND TRUTH AND RECONSTRUCTION

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 5. SAVE GROUND TRUTH AND RECONSTRUCTION

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# ============================================================

# Ground truth: dict with all true data
gt_data = {
    'wavenumber': wavenumber,
    'pure_components': pure_components,
    'true_weights': true_weights,
    'clean_spectra': clean_spectra,
    'baselines': baselines,
    'observed_spectra': observed_spectra,
}
np.save('results/ground_truth.npy', gt_data, allow_pickle=True)

# Reconstruction: dict with all recovered data
recon_data = {
    'corrected_spectra': corrected_spectra,
    'recovered_components': H_matched,
    'recovered_weights': W_matched,
    'fitted_baselines': fitted_baselines,
    'permutation': col_ind,
}
np.save('results/reconstruction.npy', recon_data, allow_pickle=True)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ============================================================

### 6. VISUALIZATION – 6 subplots

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# 6. VISUALIZATION – 6 subplots

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

### (a) Example mixed spectrum: observed vs corrected vs clean ---

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# --- (a) Example mixed spectrum: observed vs corrected vs clean ---
ax = axes[0, 0]
idx = 0  # first mixture
ax.plot(wavenumber, observed_spectra[idx], 'b-', alpha=0.6, label='Observed (with baseline+noise)')
ax.plot(wavenumber, corrected_spectra[idx], 'r-', linewidth=1.5, label='Baseline corrected')
ax.plot(wavenumber, clean_spectra[idx], 'k--', linewidth=1.5, label='Ground truth (clean)')
ax.set_xlabel('Wavenumber (cm⁻¹)')
ax.set_ylabel('Intensity')
ax.set_title('(a) Baseline Correction Example')
ax.legend(fontsize=8)

### (b) Baseline correction detail ---

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# --- (b) Baseline correction detail ---
ax = axes[0, 1]
ax.plot(wavenumber, observed_spectra[idx], 'b-', alpha=0.5, label='Observed')
ax.plot(wavenumber, fitted_baselines[idx], 'g-', linewidth=2, label='Fitted baseline')
ax.plot(wavenumber, baselines[idx], 'm--', linewidth=2, label='True baseline')
ax.set_xlabel('Wavenumber (cm⁻¹)')
ax.set_ylabel('Intensity')
ax.set_title('(b) True vs Fitted Baseline')
ax.legend(fontsize=8)

### (c) True vs recovered component spectra ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- (c) True vs recovered component spectra ---
ax = axes[0, 2]
colors = ['#e41a1c', '#377eb8', '#4daf4a']
labels = ['Mineral A', 'Mineral B', 'Mineral C']
for i in range(n_components):
    ax.plot(wavenumber, pure_components[i], '-', color=colors[i],
            linewidth=2, label=f'True {labels[i]}')
    ax.plot(wavenumber, H_matched[i], '--', color=colors[i],
            linewidth=1.5, alpha=0.8, label=f'Recovered (CC={comp_cc_list[i]:.3f})')
ax.set_xlabel('Wavenumber (cm⁻¹)')
ax.set_ylabel('Normalized Intensity')
ax.set_title('(c) True vs Recovered Components')
ax.legend(fontsize=7, ncol=2)

### (d) Mixing proportions: true vs recovered (scatter) ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- (d) Mixing proportions: true vs recovered (scatter) ---
ax = axes[1, 0]
for i in range(n_components):
    ax.scatter(true_weights[:, i], W_matched[:, i], color=colors[i],
               s=40, label=f'{labels[i]}', alpha=0.8, edgecolors='k', linewidth=0.3)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Ideal')
ax.set_xlabel('True Mixing Proportion')
ax.set_ylabel('Recovered Mixing Proportion')
ax.set_title('(d) Mixing Proportions: True vs Recovered')
ax.legend(fontsize=8)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_aspect('equal')

### (e) Residual after baseline correction ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- (e) Residual after baseline correction ---
ax = axes[1, 1]
residuals = corrected_spectra - clean_spectra
for i in range(min(5, n_mixtures)):
    ax.plot(wavenumber, residuals[i], alpha=0.5, linewidth=0.8, label=f'Mix {i+1}')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_xlabel('Wavenumber (cm⁻¹)')
ax.set_ylabel('Residual')
ax.set_title(f'(e) Baseline Correction Residuals (PSNR={psnr_baseline:.1f} dB)')
ax.legend(fontsize=7)

### (f) Per-mixture weight recovery bar chart ---

Intermediate processing step in the pipeline.

In [ ]:
# --- (f) Per-mixture weight recovery bar chart ---
ax = axes[1, 2]
mix_indices = np.arange(n_mixtures)
bar_width = 0.12
for i in range(n_components):
    ax.bar(mix_indices - bar_width + i * bar_width, true_weights[:, i],
           bar_width, color=colors[i], alpha=0.5, label=f'True {labels[i]}')
    ax.bar(mix_indices + i * bar_width, W_matched[:, i],
           bar_width, color=colors[i], edgecolor='k', linewidth=0.5,
           label=f'Rec {labels[i]}')
ax.set_xlabel('Mixture Index')
ax.set_ylabel('Weight')
ax.set_title('(f) Weight Recovery per Mixture')
# Simplify legend
handles, lbls = ax.get_legend_handles_labels()
ax.legend(handles[:6], lbls[:6], fontsize=6, ncol=2)
ax.set_xticks(mix_indices)

plt.suptitle('Raman Spectral Unmixing & Baseline Correction (rampy + NMF)',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('results/reconstruction_result.png', dpi=200, bbox_inches='tight')
plt.close()

print("\nVisualization saved to results/reconstruction_result.png")
print("Metrics saved to results/metrics.json")
print("Ground truth saved to results/ground_truth.npy")
print("Reconstruction saved to results/reconstruction.npy")
print("\n=== TASK COMPLETE ===")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 5. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **rampy_unmix**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=21.99 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Rampy: Raman spectroscopy analysis
- Repository: https://github.com/charlesll/rampy
